OUTAGE DETECTED: getting weather data when you know the location

In [1]:
import requests
from requests.exceptions import HTTPError, Timeout, RequestException

def make_nws_request(endpoint, user_agent):
    headers = {
        "User-Agent": user_agent,
    }

    try:
        response = requests.get(
                       endpoint, 
                       headers=headers
                   )
        # Raise HTTPError for bad responses (4xx or 5xx)
        response.raise_for_status()
        return response.json()

    except HTTPError as http_err:
        print(f"HTTP error occurred: {http_err} - Status code: {response.status_code}")
    except Timeout as timeout_err:
        print(f"Request timed out: {timeout_err}")
    except RequestException as req_err:
        print(f"Request error: {req_err}")
    
    return None  # Return None if an error occurred


if __name__ == "__main__":
    # Sample user agent
    user_agent = "MyWeatherApp/1.0"

    BASE_URL = "https://api.weather.gov"
    
    # Example 1: Get forecast for a specific location
    lat, lon = 39.7456, -9.0892
    forecast_url = f"{BASE_URL}/points/{lat},{lon}"
    data = make_nws_request(forecast_url, user_agent)
    
    if data:
        print(data)
    else:
        print("Failed to retrieve data.")

HTTP error occurred: 404 Client Error: Not Found for url: https://api.weather.gov/points/39.7456,-9.0892 - Status code: 404
Failed to retrieve data.


In [2]:
import requests
# Define the API endpoint and parameters
endpoint = "https://api.weather.gov/gridpoints/ILX/32,53/forecast"
headers = {
   "User-Agent": "MyWeatherApp (myemail@example.com)",
   "Accept": "application/geo+json"
}
# Make the request to the API
response = requests.get(endpoint, headers=headers)
# Check if the request was successful
if response.status_code == 200:
   # Parse the JSON response
   data = response.json()
   # Extract the current weather conditions
   current_conditions = data["properties"]["periods"][0]
   print(f"Current temperature: {current_conditions['temperature']} {current_conditions['temperatureUnit']}")
   print(f"Forecast: {current_conditions['shortForecast']}")
else:
   print(f"Error: {response.status_code}")

Current temperature: 85 F
Forecast: Chance Showers And Thunderstorms


In [3]:
print(f"Current temperature: {current_conditions}")

Current temperature: {'number': 1, 'name': 'Today', 'startTime': '2025-09-20T11:00:00-05:00', 'endTime': '2025-09-20T18:00:00-05:00', 'isDaytime': True, 'temperature': 85, 'temperatureUnit': 'F', 'temperatureTrend': '', 'probabilityOfPrecipitation': {'unitCode': 'wmoUnit:percent', 'value': 30}, 'windSpeed': '6 mph', 'windDirection': 'ESE', 'icon': 'https://api.weather.gov/icons/land/day/tsra_hi,20/tsra_hi,30?size=medium', 'shortForecast': 'Chance Showers And Thunderstorms', 'detailedForecast': 'A chance of showers and thunderstorms. Partly sunny. High near 85, with temperatures falling to around 80 in the afternoon. East southeast wind around 6 mph. Chance of precipitation is 30%. New rainfall amounts less than a tenth of an inch possible.'}


WEATHER MONITORING: returning weather information and location
* second block is more useful but 3rd block has best response it just takes a while to load cuz it filters out test messages

In [4]:
import requests

headers = {'User-Agent' : 'myapp'}
endpoint = 'https://api.weather.gov/alerts/active'

response = requests.get(endpoint, headers = headers)
data = response.json()

# `features` contains the data we want.
print(data) 

{'@context': ['https://geojson.org/geojson-ld/geojson-context.jsonld', {'@version': '1.1', 'wx': 'https://api.weather.gov/ontology#', '@vocab': 'https://api.weather.gov/ontology#'}], 'type': 'FeatureCollection', 'features': [{'id': 'https://api.weather.gov/alerts/urn:oid:2.49.0.1.840.0.2c1cf0daca1fb82153775c22abcb03b61b71888b.001.1', 'type': 'Feature', 'geometry': None, 'properties': {'@id': 'https://api.weather.gov/alerts/urn:oid:2.49.0.1.840.0.2c1cf0daca1fb82153775c22abcb03b61b71888b.001.1', '@type': 'wx:Alert', 'id': 'urn:oid:2.49.0.1.840.0.2c1cf0daca1fb82153775c22abcb03b61b71888b.001.1', 'areaDesc': 'Coastal Waters From Cape Flattery To James Island Out 10 Nm; Coastal Waters From James Island To Point Grenville Out 10 Nm; Coastal Waters From Point Grenville To Cape Shoalwater Out 10 Nm; Coastal Waters From Cape Flattery To James Island 10 To 60 Nm; Waters From James Island To Point Grenville 10 To 60 Nm; Coastal Waters From Point Grenville To Cape Shoalwater 10 To 60 Nm', 'geocode'

In [5]:
import requests

# Fetch active severe weather alerts
headers = {'User-Agent' : 'myapp'}
endpoint = 'https://api.weather.gov/alerts/active'

response = requests.get(endpoint, headers = headers)
data = response.json()

# Extract relevant data from alerts
alerts_info = []
for feature in data.get("features", []):
    properties = feature.get("properties", {})
    geometry = feature.get("geometry")

    alert_id = feature.get("id")
    event = properties.get("event")
    description = properties.get("description", "")
    area_desc = properties.get("areaDesc", "")
    effective = properties.get("effective")
    expires = properties.get("expires")
    severity = properties.get("severity")
    certainty = properties.get("certainty")
    urgency = properties.get("urgency")

    # Extract coordinates if available
    region_coords = None
    if geometry and "coordinates" in geometry:
        region_coords = geometry["coordinates"]

    alert_info = {
        "id": alert_id,
        "event": event,
        "reason": description[:300],  # Trim long text
        "area": area_desc,
        "severity": severity,
        "urgency": urgency,
        "certainty": certainty,
        "effective": effective,
        "expires": expires,
        "region_coords": region_coords
    }
    alerts_info.append(alert_info)

# Print the first 3 alerts for preview
for alert in alerts_info[:3]:
    print(f"Event: {alert['event']}")
    print(f"Area: {alert['area']}")
    print(f"Reason: {alert['reason']}")
    print(f"Coords: {alert['region_coords'][:1] if alert['region_coords'] else 'N/A'}")
    print("-" * 40)


Event: Small Craft Advisory
Area: Coastal Waters From Cape Flattery To James Island Out 10 Nm; Coastal Waters From James Island To Point Grenville Out 10 Nm; Coastal Waters From Point Grenville To Cape Shoalwater Out 10 Nm; Coastal Waters From Cape Flattery To James Island 10 To 60 Nm; Waters From James Island To Point Grenville 10 To 60 Nm; Coastal Waters From Point Grenville To Cape Shoalwater 10 To 60 Nm
Reason: * WHAT...South winds 15 to 25 kt.

* WHERE...Coastal Waters from Cape Flattery to Cape Shoalwater
out to 60 nm.

* WHEN...From 2 PM this afternoon to 2 AM PDT Sunday.

* IMPACTS...Conditions will be hazardous to small craft.
Coords: N/A
----------------------------------------
Event: Small Craft Advisory
Area: Central U.S. Waters Strait Of Juan De Fuca; East Entrance U.S. Waters Strait Of Juan De Fuca
Reason: * WHAT...Southwest winds 20 to 30 kt.

* WHERE...Central U. S. Waters Strait Of Juan De Fuca and East
Entrance U. S. Waters Strait Of Juan De Fuca.

* WHEN...From 11 PM

In [6]:
import requests

# --- Config ---
API_BASE = "https://api.weather.gov"
# HEADERS = {
#     "User-Agent": "MyWeatherAlertSystem (you@example.com)"  # Replace with your info
# }

# # --- Step 1: Fetch active severe alerts ---
# params = {
#     "severity": "severe",
#     "urgency": "immediate"
# }
# response = requests.get(f"{API_BASE}/alerts/active", params=params, headers=HEADERS)
# data = response.json()

import requests

# Fetch active severe weather alerts
HEADERS = {'User-Agent' : 'myapp'}
endpoint = 'https://api.weather.gov/alerts/active'

response = requests.get(endpoint, headers = HEADERS)
data = response.json()

alerts_info = []

for feature in data.get("features", []):
    properties = feature.get("properties", {})
    geometry = feature.get("geometry")
    affected_zones = properties.get("affectedZones", [])

    event = properties.get("event")
    if not event or event.lower() == "test message":
        continue  # Skip test messages

    # Extract base info
    alert_info = {
        "id": feature.get("id"),
        "event": event,
        "reason": properties.get("description", "")[:300],
        "area": properties.get("areaDesc", ""),
        "severity": properties.get("severity"),
        "urgency": properties.get("urgency"),
        "certainty": properties.get("certainty"),
        "effective": properties.get("effective"),
        "expires": properties.get("expires"),
        "region_coords": None
    }

    # --- Step 2: Use primary geometry if available ---
    if geometry and "coordinates" in geometry:
        alert_info["region_coords"] = geometry["coordinates"]
    else:
        # --- Step 3: Fallback: Try fetching geometry for first affected zone ---
        if affected_zones:
            zone_id = affected_zones[0].split("/")[-1]  # Extract zone ID
            zone_type = affected_zones[0].split("/")[-2]  # e.g. "forecast", "county"
            zone_url = f"{API_BASE}/zones/{zone_type}/{zone_id}"

            zone_response = requests.get(zone_url, headers=HEADERS)
            if zone_response.status_code == 200:
                zone_data = zone_response.json()
                zone_geom = zone_data.get("geometry")
                if zone_geom and "coordinates" in zone_geom:
                    alert_info["region_coords"] = zone_geom["coordinates"]

    alerts_info.append(alert_info)

# --- Step 4: Display summary ---
for alert in alerts_info[:5]:  # limit for brevity
    print(f"Event: {alert['event']}")
    print(f"Area: {alert['area']}")
    print(f"Reason: {alert['reason']}")
    if alert['region_coords']:
        print(f"Coords: {alert['region_coords'][:1]}...")  # preview only
    else:
        print("Coords: N/A")
    print("-" * 40)


Event: Small Craft Advisory
Area: Coastal Waters From Cape Flattery To James Island Out 10 Nm; Coastal Waters From James Island To Point Grenville Out 10 Nm; Coastal Waters From Point Grenville To Cape Shoalwater Out 10 Nm; Coastal Waters From Cape Flattery To James Island 10 To 60 Nm; Waters From James Island To Point Grenville 10 To 60 Nm; Coastal Waters From Point Grenville To Cape Shoalwater 10 To 60 Nm
Reason: * WHAT...South winds 15 to 25 kt.

* WHERE...Coastal Waters from Cape Flattery to Cape Shoalwater
out to 60 nm.

* WHEN...From 2 PM this afternoon to 2 AM PDT Sunday.

* IMPACTS...Conditions will be hazardous to small craft.
Coords: [[[-124.7456969, 48.4960011], [-124.7302736, 48.382743], [-124.7312469, 48.382984099999994], [-124.7314147, 48.3827095], [-124.7302856, 48.382118199999994], [-124.7301483, 48.38182059999999], [-124.7296981, 48.38152309999999], [-124.72973599999999, 48.38122599999999], [-124.73052209999999, 48.38097379999999], [-124.73131489999999, 48.381316799999

In [ ]:

def get_urgent_areas(state):
    # Fetch active severe weather alerts
    HEADERS = {'User-Agent' : 'myapp'}
    endpoint = f'https://api.weather.gov/alerts/active?area={state}'

    response = requests.get(endpoint, headers=HEADERS)
    data = response.json()
    
    urgent_zones = []

    for feature in data.get("features", []):
        properties = feature.get("properties", {})
        affected_zones = properties.get("affectedZones", [])
        
        if alert_info['urgency'] != "Immediate": continue
        
        event = properties.get("event")
        if not event or event.lower() == "test message":
            continue  # Skip test messages
        
        alert_info = {
            "id": feature.get("id"),
            "event": event,
            "reason": properties.get("description", "")[:300],
            "affected_zones": affected_zones,
            "severity": properties.get("severity"),
            "urgency": properties.get("urgency"),
            "certainty": properties.get("certainty"),
            "effective": properties.get("effective"),
            "expires": properties.get("expires")
        }
        
        urgent_zones.append(alert_info)
    
    return urgent_zones


        # Extract base info
        

        # print(alert_info)
        # --- Step 2: Use primary geometry if available ---
        # if geometry and "coordinates" in geometry:
        #     alert_info["region_coords"] = geometry["coordinates"]
        # else:
        #     # --- Step 3: Fallback: Try fetching geometry for first affected zone ---
        #     if affected_zones:
        #         zone_id = affected_zones[0].split("/")[-1]  # Extract zone ID
        #         zone_type = affected_zones[0].split("/")[-2]  # e.g. "forecast", "county"
        #         zone_url = f"{API_BASE}/zones/{zone_type}/{zone_id}"

        #         zone_response = requests.get(zone_url, headers=HEADERS)
        #         if zone_response.status_code == 200:
        #             zone_data = zone_response.json()
        #             zone_geom = zone_data.get("geometry")
        #             if zone_geom and "coordinates" in zone_geom:
        #                 alert_info["region_coords"] = zone_geom["coordinates"]

        # alerts_info.append(alert_info)
    
    # --- Step 4: Display summary ---
    # for alert in alerts_info[:5]:  # limit for brevity
    #     print(f"Event: {alert['event']}")
    #     print(f"Area: {alert['area']}")
    #     print(f"Reason: {alert['reason']}")
    #     if alert['region_coords']:
    #         print(f"Coords: {alert['region_coords'][:1]}...")  # preview only
    #     else:
    #         print("Coords: N/A")
    #     print("-" * 40)

get_urgent_alerts('TX')


None
['https://api.weather.gov/zones/forecast/TXZ102', 'https://api.weather.gov/zones/forecast/TXZ103', 'https://api.weather.gov/zones/forecast/TXZ104', 'https://api.weather.gov/zones/forecast/TXZ105', 'https://api.weather.gov/zones/forecast/TXZ117', 'https://api.weather.gov/zones/forecast/TXZ118', 'https://api.weather.gov/zones/forecast/TXZ119', 'https://api.weather.gov/zones/forecast/TXZ120', 'https://api.weather.gov/zones/forecast/TXZ121', 'https://api.weather.gov/zones/forecast/TXZ131', 'https://api.weather.gov/zones/forecast/TXZ133', 'https://api.weather.gov/zones/forecast/TXZ134', 'https://api.weather.gov/zones/forecast/TXZ135']


In [20]:
import requests
import math
from typing import List, Tuple, Dict, Any

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Calculate the great circle distance between two points on Earth in kilometers.
    """
    R = 6371  # Earth's radius in kilometers
    
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    
    return R * c

def polygon_to_circle(coordinates: List[List[List[float]]]) -> Dict[str, float]:
    """
    Convert a polygon to a circle by finding the centroid and maximum distance to any vertex.
    
    Args:
        coordinates: GeoJSON polygon coordinates (array of rings, each ring is array of [lon, lat] points)
    
    Returns:
        Dict with 'lat', 'lon' (center), and 'radius_km' (radius in kilometers)
    """
    if not coordinates or not coordinates[0]:
        raise ValueError("Invalid coordinates")
    
    # Get the outer ring (first ring in the coordinates)
    outer_ring = coordinates[0]
    
    # Calculate centroid
    total_lat = sum(point[1] for point in outer_ring)  # lat is index 1
    total_lon = sum(point[0] for point in outer_ring)  # lon is index 0
    num_points = len(outer_ring)
    
    center_lat = total_lat / num_points
    center_lon = total_lon / num_points
    
    # Find maximum distance from center to any vertex
    max_distance = 0
    for point in outer_ring:
        distance = haversine_distance(center_lat, center_lon, point[1], point[0])
        max_distance = max(max_distance, distance)
    
    return {
        'lat': center_lat,
        'lon': center_lon,
        'radius_km': max_distance
    }

def fetch_zone_geometry(zone_url: str, headers: Dict[str, str]) -> Dict[str, float]:
    """
    Fetch zone geometry from NWS API and convert to circle.
    
    Args:
        zone_url: URL to the zone endpoint
        headers: HTTP headers for the request
    
    Returns:
        Dict with 'lat', 'lon', and 'radius_km'
    """
    try:
        response = requests.get(zone_url, headers=headers)
        response.raise_for_status()
        zone_data = response.json()
        
        geometry = zone_data.get('geometry', {})
        coordinates = geometry.get('coordinates', [])
        
        if not coordinates:
            raise ValueError(f"No coordinates found for zone {zone_url}")
        
        return polygon_to_circle(coordinates)
    
    except Exception as e:
        print(f"Error fetching zone {zone_url}: {e}")
        return None

def get_urgent_areas(state: str) -> List[Dict[str, Any]]:
    """
    Fetch active severe weather alerts and convert affected zones to center points and radii.
    
    Args:
        state: Two-letter state code
        
    Returns:
        List of alert info dictionaries with added geometry data
    """
    HEADERS = {'User-Agent': 'myapp'}
    endpoint = f'https://api.weather.gov/alerts/active?area={state}'
    
    try:
        response = requests.get(endpoint, headers=HEADERS)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Error fetching alerts: {e}")
        return []
    
    urgent_zones = []
    
    for feature in data.get("features", []):
        properties = feature.get("properties", {})
        
        # Check urgency first
        # if properties.get('urgency') != "Immediate":
            # continue
        
        event = properties.get("event")
        if not event or event.lower() == "test message":
            continue
        
        affected_zones = properties.get("affectedZones", [])
        
        alert_info = {
            "id": feature.get("id"),
            "event": event,
            "reason": properties.get("description", "")[:300],
            "affected_zones": affected_zones,
            "severity": properties.get("severity"),
            "urgency": properties.get("urgency"),
            "certainty": properties.get("certainty"),
            "effective": properties.get("effective"),
            "expires": properties.get("expires"),
            "zone_circles": []  # Will store the converted circle data
        }
        
        # Convert each affected zone to a circle
        for zone_url in affected_zones:
            circle_data = fetch_zone_geometry(zone_url, HEADERS)
            if circle_data:
                # Add zone name from URL for reference
                zone_id = zone_url.split('/')[-1] if '/' in zone_url else zone_url
                circle_data['zone_id'] = zone_id
                alert_info["zone_circles"].append(circle_data)
        
        urgent_zones.append(alert_info)
    
    return urgent_zones

# Example usage:
if __name__ == "__main__":
    # Test fetching alerts
    alerts = get_urgent_areas("TX")
    print(alerts)
    for alert in alerts:
        print(f"\nAlert: {alert['event']}")
        print(f"Affected zones converted to circles: {len(alert['zone_circles'])}")
        for circle in alert['zone_circles']:
            print(f"  Zone {circle['zone_id']}: Center ({circle['lat']:.4f}, {circle['lon']:.4f}), Radius: {circle['radius_km']:.1f} km")

[{'id': 'https://api.weather.gov/alerts/urn:oid:2.49.0.1.840.0.71af3c1fc604d47540f5e6717e378ac0aced1d35.001.1', 'event': 'Air Quality Alert', 'reason': 'AQAFWD\n\nThe Texas Commission on Environmental Quality (TCEQ) has issued\nan Ozone Action Day for the Dallas-Fort Worth area for Saturday,\nSeptember 20, 2025.\n\nAtmospheric conditions are expected to be favorable for producing\nhigh levels of ozone air pollution in the Dallas-Fort Worth area\non Saturday', 'affected_zones': ['https://api.weather.gov/zones/forecast/TXZ102', 'https://api.weather.gov/zones/forecast/TXZ103', 'https://api.weather.gov/zones/forecast/TXZ104', 'https://api.weather.gov/zones/forecast/TXZ105', 'https://api.weather.gov/zones/forecast/TXZ117', 'https://api.weather.gov/zones/forecast/TXZ118', 'https://api.weather.gov/zones/forecast/TXZ119', 'https://api.weather.gov/zones/forecast/TXZ120', 'https://api.weather.gov/zones/forecast/TXZ121', 'https://api.weather.gov/zones/forecast/TXZ131', 'https://api.weather.gov/zo